# Ecological Scarcity 2021: from one score to its hotspots

**How many eco-points does 1 tonne-kilometre of freight transport cause, and where do they come from?**

Use the Swiss Ecological Scarcity 2021 method (UBP, version 1.06) to calculate a score, then explore the contributing flows, processes and locations—just as in the AWARE example.

**Choose an activity → calculate → explore the charts.** The full breakdown into 20 indicators is optional.

You need Edges, Matplotlib, pandas and an existing Brightway project containing **ecoinvent 3.11 cutoff**. This example uses its Swiss market for FAME-powered 28-tonne lorry transport. No inventory import is needed.

## 1. Choose your activity

Edit the settings below to match your local project. `AMOUNT` is the quantity of the selected activity's reference product; the default is **1 ton kilometer** (1 tonne transported over 1 km).

In [ ]:
import bw2data
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from textwrap import fill

from edges import EdgeLCIA, get_available_methods

PROJECT = "ecoinvent-3.11-cutoff"
DATABASE = "ecoinvent-3.11-cutoff"
ACTIVITY_NAME = (
    "market for transport, freight, lorry 28 metric ton, "
    "fatty acid methyl ester 100%"
)
REFERENCE_PRODUCT = ACTIVITY_NAME.removeprefix("market for ")
LOCATION = "CH"
UNIT = "ton kilometer"
AMOUNT = 1.0

plt.rcParams.update({"font.size": 11, "axes.spines.top": False,
                     "axes.spines.right": False})

In [ ]:
if PROJECT not in {project.name for project in bw2data.projects}:
    raise ValueError(f"Project {PROJECT!r} not found. Set PROJECT to an existing project.")
bw2data.projects.set_current(PROJECT, writable=False, update=False)
if DATABASE not in bw2data.databases:
    raise ValueError(f"Database {DATABASE!r} not found. Check DATABASE in the settings.")

matches = [
    a for a in bw2data.Database(DATABASE)
    if a.get("name") == ACTIVITY_NAME
    and a.get("reference product") == REFERENCE_PRODUCT
    and a.get("location") == LOCATION
    and a.get("unit") == UNIT
]
if len(matches) != 1:
    raise ValueError(
        f"Found {len(matches)} activities. Check the exact name, product, location and unit."
    )
activity = matches[0]
functional_unit = {activity: AMOUNT}
fu_label = f"{AMOUNT:g} {activity['unit']} · {activity['location']}"
display(Markdown(f"**Selected:** {activity['name']}\n\n**Functional unit:** {fu_label}"))

## 2. Calculate the eco-points

All indicators share the same unit, **UBP (eco-points)**, so they can be added into one total. A higher score represents a greater burden under the method's Swiss environmental targets. It is not a carbon-footprint score.

This is the usual Edges workflow. The method's strategies take care of exchange matching and geographic fallback.

In [ ]:
lca = EdgeLCIA(
    demand=functional_unit,
    method=("Ecological Scarcity 2021", "total", "UBP"),
)
lca.lci()
lca.apply_strategies()
lca.evaluate_cfs()
lca.lcia()

display(Markdown(f"### {lca.score:,.2f} UBP\nFor **{fu_label}**"))

## 3. What contributes most?

The reporting table connects each characterized flow or service (**supplier**) to the process using or emitting it (**consumer**). We use it to view the same total from three angles.

Blue bars are positive contributions; orange bars are negative contributions (credits). Each chart keeps the eight largest contributions by absolute size and combines the rest into **Other**, preserving the total.

In [ ]:
cf = lca.generate_cf_table(include_unmatched=False)
np.testing.assert_allclose(cf["impact"].sum(), lca.score, rtol=1e-8, atol=1e-8)

# Include the location so identically named processes in different regions stay separate.
cf["process"] = (
    cf["consumer name"].fillna("Unknown process")
    + " [" + cf["consumer location"].fillna("Unknown") + "]"
)

def contribution_chart(ax, values, title, top=8):
    values = values.reindex(values.abs().sort_values(ascending=False).index)
    shown = values.iloc[:top].copy()
    if len(values) > top:
        shown.loc["Other (remaining contributions)"] = values.iloc[top:].sum()
    positions = np.arange(len(shown))
    bars = ax.barh(positions, shown.values,
                   color=["#2878B5" if value >= 0 else "#D97732" for value in shown])
    ax.set_yticks(positions, [fill(str(label), 38) for label in shown.index])
    ax.invert_yaxis()
    ax.bar_label(bars, fmt="{:,.2f}", padding=4, fontsize=9)
    ax.axvline(0, color="0.5", linewidth=0.8)
    ax.set(title=title, xlabel="Contribution [UBP for the selected functional unit]")
    ax.margins(x=0.25)
    ax.grid(axis="x", alpha=0.15)
    ax.set_axisbelow(True)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 19), layout="constrained")
for ax, column, title in zip(
    axes,
    ["supplier name", "process", "consumer location"],
    ["Which flows and services?", "Which processes?", "Which locations?"],
):
    contributions = cf.groupby(column, dropna=False)["impact"].sum()
    contribution_chart(ax, contributions, title)
fig.suptitle(f"Ecological Scarcity 2021 · {lca.score:,.2f} UBP\n{fu_label}",
             fontsize=17, fontweight="bold")
plt.show()

**How to read these charts**

- These are three views of the **same score**; do not add the panels together.
- Process contributions are the impacts assigned to exchanges at each process, not that process's full upstream footprint.
- Locations are the **consumer locations in the inventory**. Regional labels such as RER, RoW and GLO remain visible; they are not individual countries or necessarily the places where damage occurs.
- A small net contribution can hide positive and negative exchanges that cancel. The table below lets you inspect individual exchanges.

Only four indicators contain geographic rules: **Land use**, **Main air pollutants and particulates**, **Water pollutants**, and **Water resources, net balance**. A location contribution chart therefore does not mean that every eco-factor is regionalized.

### A closer look at the largest exchanges

`amount × CF = impact`: the amount is scaled to your functional unit. Its unit belongs to the supplier flow or service; impacts are always in UBP.

In [ ]:
columns = ["supplier name", "consumer name", "consumer location",
           "amount", "CF", "impact"]
top_exchanges = cf.loc[cf["impact"].abs().nlargest(8).index, columns]
display(top_exchanges.reset_index(drop=True).style.format(
    {"amount": "{:.3g}", "CF": "{:,.3g}", "impact": "{:,.2f}"}
).bar(subset=["impact"], color=["#D97732", "#2878B5"], align="zero"))

### Where does Noise enter the score?

Most indicators characterize environmental flows. **Noise** characterizes transport-service exchanges instead. Edges includes both in the total; no separate Noise calculation is needed.

In [ ]:
components = pd.Series({
    "Environmental flows": lca.score_by_matrix.get("biosphere", 0.0),
    "Transport noise": lca.score_by_matrix.get("technosphere", 0.0),
})
fig, ax = plt.subplots(figsize=(10, 3), layout="constrained")
contribution_chart(ax, components, "Two components of the total")
plt.show()

## 4. Optional: break the score down by indicator

Want to distinguish climate, water, land use and the other indicators? Set `RUN_CATEGORIES = True` and run the next two cells. This performs **20 additional LCA calculations** and can take several minutes. Leave it off for the shorter walkthrough.

All indicator scores are in UBP. Their sum should reproduce the total above.

In [ ]:
RUN_CATEGORIES = False
category_scores = None

if RUN_CATEGORIES:
    category_methods = sorted(
        method for method in get_available_methods()
        if method[0] == "Ecological Scarcity 2021" and method[1] != "total"
    )
    results = {}
    for index, method in enumerate(category_methods, start=1):
        print(f"{index}/{len(category_methods)} · {method[1]}")
        category_lca = EdgeLCIA(demand=functional_unit, method=method)
        category_lca.lci()
        category_lca.apply_strategies()
        category_lca.evaluate_cfs()
        category_lca.lcia()
        results[method[1]] = float(category_lca.score)
        del category_lca
    category_scores = pd.Series(results, name="UBP")
    np.testing.assert_allclose(category_scores.sum(), lca.score, rtol=1e-8, atol=1e-8)
else:
    print("Skipped. Set RUN_CATEGORIES = True to explore all indicators.")

In [ ]:
if category_scores is not None:
    fig, ax = plt.subplots(figsize=(12, 11), layout="constrained")
    contribution_chart(ax, category_scores, "Which indicators drive the score?", top=20)
    plt.show()

## Try another activity

Change the activity settings in section 1 and run the notebook again. Keep the functional unit consistent when comparing alternatives. Results depend on the activity, database version and system model.

For a single indicator, replace `"total"` in the method tuple with, for example, `"Global warming"`, `"Land use"` or `"Noise"`, using a separate `EdgeLCIA` object.

For deeper matching diagnostics, use `lca.statistics()` or `lca.generate_cf_table(include_unmatched=True)`. An unmatched exchange is not automatically an error: it can fall outside the method's scope.

Method construction and release checks are documented in [the UBP developer notes](../../dev/UBP/README.md).